<a href="https://colab.research.google.com/github/greensky0107/self_study/blob/main/%ED%8C%8C%EC%9D%B4%EC%8D%AC_%EA%B8%B0%EC%B4%88_%ED%86%B5%EA%B3%84%EB%9F%89_%EA%B3%84%EC%82%B0_%EC%BD%94%EB%93%9C_(Pandas).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import pandas as pd
import numpy as np

# FDS 분석을 위한 예시 금융 거래 데이터프레임 생성
# account_id: 계좌 번호
# transaction_amount: 거래 금액
data = {
    'account_id': ['A01', 'A01', 'B02', 'A01', 'C03', 'B02', 'A01', 'C03', 'B02', 'A01'],
    'transaction_amount': [10000, 5000, 300000, 7000, 15000, 50000, 12000, 80000, 45000, 9000],
    'transaction_type': ['출금', '출금', '입금', '출금', '출금', '입금', '출금', '입금', '입금', '출금']
}
df = pd.DataFrame(data)

print("--- 1. 원본 데이터 ---")
print(df)
print("-" * 30)

--- 1. 원본 데이터 ---
  account_id  transaction_amount transaction_type
0        A01               10000               출금
1        A01                5000               출금
2        B02              300000               입금
3        A01                7000               출금
4        C03               15000               출금
5        B02               50000               입금
6        A01               12000               출금
7        C03               80000               입금
8        B02               45000               입금
9        A01                9000               출금
------------------------------


In [6]:
# --- 전체 데이터에 대한 기초 통계량 ---

# 2. describe() 함수로 요약 통계량 한 번에 확인하기
# 데이터 개수(count), 평균(mean), 표준편차(std), 최소값(min),
# 25%(1사분위), 50%(중앙값), 75%(3사분위), 최대값(max)을 보여줍니다.
summary = df['transaction_amount'].describe()
print("\n--- 2. 거래 금액 요약 통계량 (describe) ---")
print(summary)
print("-" * 30)


--- 2. 거래 금액 요약 통계량 (describe) ---
count        10.000000
mean      53300.000000
std       90148.088043
min        5000.000000
25%        9250.000000
50%       13500.000000
75%       48750.000000
max      300000.000000
Name: transaction_amount, dtype: float64
------------------------------


In [7]:
# 3. 개별 기초 통계량 계산하기
print("\n--- 3. 개별 기초 통계량 ---")
mean_val = df['transaction_amount'].mean()
median_val = df['transaction_amount'].median()
# 최빈값은 여러 개일 수 있어 Series 형태로 반환되므로, 첫 번째 값을 선택합니다.
mode_val = df['transaction_amount'].mode()[0]
std_val = df['transaction_amount'].std()
var_val = df['transaction_amount'].var()
range_val = df['transaction_amount'].max() - df['transaction_amount'].min()
skew_val = df['transaction_amount'].skew()
kurt_val = df['transaction_amount'].kurt()
# 사분위수 및 IQR(사분위수 범위)
q1 = df['transaction_amount'].quantile(0.25)
q3 = df['transaction_amount'].quantile(0.75)
iqr = q3 - q1

print(f"평균 (Mean): {mean_val:,.0f}원")
print(f"중앙값 (Median): {median_val:,.0f}원")
print(f"최빈값 (Mode): {mode_val:,.0f}원")
print(f"표준편차 (Standard Deviation): {std_val:,.2f}")
print(f"분산 (Variance): {var_val:,.2f}")
print(f"범위 (Range): {range_val:,.0f}원")
print(f"왜도 (Skewness): {skew_val:.2f}")
print(f"첨도 (Kurtosis): {kurt_val:.2f}")
print(f"사분위수 범위 (IQR): {iqr:,.0f}원")
print("-" * 30)





--- 3. 개별 기초 통계량 ---
평균 (Mean): 53,300원
중앙값 (Median): 13,500원
최빈값 (Mode): 5,000원
표준편차 (Standard Deviation): 90,148.09
분산 (Variance): 8,126,677,777.78
범위 (Range): 295,000원
왜도 (Skewness): 2.75
첨도 (Kurtosis): 7.98
사분위수 범위 (IQR): 39,500원
------------------------------


In [8]:
# --- 그룹별 통계량 (FDS 피쳐 생성의 핵심) ---

# 4. 계좌(account_id)별로 그룹화하여 통계량 계산하기
# FDS 모델의 피쳐로 사용하기 위해 각 계좌의 거래 패턴을 파악합니다.
# .agg() 함수를 사용하면 여러 통계량을 동시에 계산할 수 있습니다.
grouped_stats = df.groupby('account_id')['transaction_amount'].agg(
    [
        ('거래횟수', 'count'),
        ('총거래액', 'sum'),
        ('평균거래액', 'mean'),
        ('거래액표준편차', 'std'),
        ('최대거래액', 'max'),
        ('최소거래액', 'min')
    ]
).reset_index()

# NaN(Not a Number) 값을 0으로 채웁니다. (거래가 1건인 경우 표준편차가 NaN으로 계산됨)
grouped_stats = grouped_stats.fillna(0)

print("\n--- 4. 계좌별 거래 통계량 (피쳐 생성 예시) ---")
print(grouped_stats)
print("-" * 30)


--- 4. 계좌별 거래 통계량 (피쳐 생성 예시) ---
  account_id  거래횟수    총거래액          평균거래액        거래액표준편차   최대거래액  최소거래액
0        A01     5   43000    8600.000000    2701.851217   12000   5000
1        B02     3  395000  131666.666667  145802.377667  300000  45000
2        C03     2   95000   47500.000000   45961.940777   80000  15000
------------------------------
